# NB_06 — Stage 4: Evaluation

Evaluates the fine-tuned Qwen2.5-VL-7B adapter on the 280-sample held-out set.
Two checkpoints are evaluated:
1. **`checkpoint-1120`** — end of epoch 2 (lowest eval loss in Run 2, recommended)
2. **`checkpoint-final`** — merged adapter from the end of epoch 3

Metrics reported: CER, WER (both via `jiwer`), exact-match rate, valid-JSON rate, avg inference time.

**Key fixes vs. Run 1 (broken eval):**
- Images are decoded from base64 data URLs and passed to the processor (`images=` arg)
- KV cache re-enabled at inference time (`model.config.use_cache = True`)
- `max_new_tokens` reduced 512 → 128 to stop runaway generation
- `repetition_penalty=1.1`, `no_repeat_ngram_size=6` added as guardrails
- `jiwer` used for CER/WER so values stay in [0, 1]

**Prerequisites:** NB_05_finetuning must have completed and saved the adapter to Drive.

## Step 6.1 — Mount Drive and set paths

In [2]:
from google.colab import drive
drive.mount('/content/drive')

import os

PROJECT_ROOT = '/content/drive/MyDrive/CUD files/NLP/NLP_Arabic_HTR_Project'
RUN_NAME     = 'run-2'
ADAPTER_DIR  = f'{PROJECT_ROOT}/models/lora_adapter/{RUN_NAME}'
EVAL_FILE    = f'{PROJECT_ROOT}/data/eval/eval.jsonl'
LOG_DIR      = f'{PROJECT_ROOT}/logs/{RUN_NAME}'
os.makedirs(LOG_DIR, exist_ok=True)

# ── Switch this to evaluate a different checkpoint ────────────────────────
# Options:
#   None            → uses checkpoint-final (merged end-of-epoch-3 adapter)
#   'checkpoint-1120' → epoch-2 checkpoint (best eval loss in Run 2)
CHECKPOINT = 'checkpoint-1120'

if CHECKPOINT:
    ADAPTER_PATH = f'{ADAPTER_DIR}/{CHECKPOINT}'
else:
    ADAPTER_PATH = f'{ADAPTER_DIR}/checkpoint-final'

print(f'Adapter path : {ADAPTER_PATH}')
print(f'Eval file    : {EVAL_FILE}')

Mounted at /content/drive
Adapter path : /content/drive/MyDrive/CUD files/NLP/NLP_Arabic_HTR_Project/models/lora_adapter/run-2/checkpoint-1120
Eval file    : /content/drive/MyDrive/CUD files/NLP/NLP_Arabic_HTR_Project/data/eval/eval.jsonl


## Step 6.2 — Install dependencies

In [3]:
import subprocess, sys

# ── Do NOT reinstall torch — Colab's pre-built CUDA version must stay ─────
# Reinstalling torch from PyPI gives a CPU-only build and breaks GPU access.

# ── Install in one shot to let pip resolve a consistent dependency set ─────
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q',
    'transformers>=4.40',   # Qwen2.5-VL support landed in 4.40
    'peft>=0.10',           # LoRA TaskType.CAUSAL_LM requires >=0.10
    'bitsandbytes>=0.43',   # NF4 double-quant support requires >=0.43
    'accelerate>=0.30',     # device_map="auto" dispatch requires >=0.30
    'qwen-vl-utils',        # process_vision_info helper (no version constraint needed)
], check=True)

# ── Verify GPU is still accessible after installs ─────────────────────────
import torch
assert torch.cuda.is_available(), (
    'CUDA not available after install. '
    'If you see this, the runtime may have been reset — re-run from the top.'
)
print(f'torch {torch.__version__}  |  CUDA {torch.version.cuda}  |  GPU: {torch.cuda.get_device_name(0)}')
print('Dependencies installed.')

torch 2.10.0+cu128  |  CUDA 12.8  |  GPU: NVIDIA A100-SXM4-40GB
Dependencies installed.


In [4]:
import subprocess, sys, torch

subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q',
    'transformers>=4.40',
    'peft>=0.10',
    'bitsandbytes>=0.43',
    'accelerate>=0.30',
    'qwen-vl-utils',
    'jiwer',
], check=True)

assert torch.cuda.is_available(), 'No GPU — switch runtime to A100.'
print(f'torch {torch.__version__}  |  GPU: {torch.cuda.get_device_name(0)}')
print('Dependencies installed.')

torch 2.10.0+cu128  |  GPU: NVIDIA A100-SXM4-40GB
Dependencies installed.


## Step 6.3 — Verify GPU

In [5]:
import torch
assert torch.cuda.is_available(), 'No GPU. Switch runtime to A100 in Runtime > Change runtime type.'
print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

GPU: NVIDIA A100-SXM4-40GB
VRAM: 42.4 GB


## Step 6.4 — Load the processor and model

The base model is loaded in 4-bit NF4, then the LoRA adapter is applied on top.
KV cache is explicitly re-enabled here — it was disabled during training for
gradient checkpointing and would otherwise remain off, causing ~10x slower inference.

In [6]:
import torch
from transformers import AutoProcessor, Qwen2_5_VLForConditionalGeneration, BitsAndBytesConfig
from peft import PeftModel

BASE_MODEL  = 'Qwen/Qwen2.5-VL-7B-Instruct'
MIN_PIXELS  = 3_136
MAX_PIXELS  = 100_352

# ── Processor ─────────────────────────────────────────────────────────────
processor = AutoProcessor.from_pretrained(
    BASE_MODEL,
    min_pixels=MIN_PIXELS,
    max_pixels=MAX_PIXELS,
)

# ── 4-bit quantized base model ────────────────────────────────────────────
bnb_config = BitsAndBytesConfig(
    load_in_4bit              = True,
    bnb_4bit_quant_type       = 'nf4',
    bnb_4bit_use_double_quant = True,
    bnb_4bit_compute_dtype    = torch.bfloat16,
)

print(f'Loading base model {BASE_MODEL}...')
base_model = Qwen2_5_VLForConditionalGeneration .from_pretrained(
    BASE_MODEL,
    quantization_config = bnb_config,
    device_map          = 'auto',
    torch_dtype         = torch.bfloat16,
    trust_remote_code   = True,
)

# ── Load LoRA adapter ─────────────────────────────────────────────────────
print(f'Loading adapter from {ADAPTER_PATH}...')
model = PeftModel.from_pretrained(base_model, ADAPTER_PATH)
model.eval()

# ── Re-enable KV cache for inference (disabled during training) ───────────
model.config.use_cache = True

print('Model and adapter loaded.')

preprocessor_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

chat_template.json: 0.00B [00:00, ?B/s]

The image processor of type `Qwen2VLImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/729 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/216 [00:00<?, ?B/s]

Loading adapter from /content/drive/MyDrive/CUD files/NLP/NLP_Arabic_HTR_Project/models/lora_adapter/run-2/checkpoint-1120...
Model and adapter loaded.


## Step 6.5 — Helper functions

In [7]:
import json, base64, io, re
from PIL import Image


def extract_images(messages):
    """Decode base64 image_url blocks from the user turn into PIL Images."""
    images = []
    for msg in messages:
        if msg['role'] != 'user':
            continue
        content = msg['content']
        if not isinstance(content, list):
            continue
        for item in content:
            if isinstance(item, dict) and item.get('type') == 'image_url':
                url = item['image_url']['url']
                b64 = url.split(',', 1)[1]
                img_bytes = base64.b64decode(b64)
                img = Image.open(io.BytesIO(img_bytes)).convert('RGB')
                images.append(img)
    return images


def get_reference(messages):
    """Extract the GPT reference transcription from the assistant turn."""
    for msg in messages:
        if msg['role'] == 'assistant':
            try:
                parsed = json.loads(msg['content'])
                return parsed.get('transcription', '').strip()
            except json.JSONDecodeError:
                return msg['content'].strip()
    return ''


def parse_output(raw_text):
    """Extract the transcription string from model output.
    Returns (transcription, is_valid_json)."""
    # Try to parse as JSON
    try:
        parsed = json.loads(raw_text)
        return parsed.get('transcription', '').strip(), True
    except json.JSONDecodeError:
        pass

    # Try to recover a partial JSON (add closing brace if missing)
    try:
        parsed = json.loads(raw_text + '}')
        return parsed.get('transcription', '').strip(), True
    except json.JSONDecodeError:
        pass

    # Fallback: regex to pull the transcription value
    m = re.search(r'"transcription"\s*:\s*"([^"]*)', raw_text)
    if m:
        return m.group(1).strip(), False

    return raw_text.strip(), False


print('Helper functions defined.')

Helper functions defined.


## Step 6.6 — Run inference on the eval set

In [8]:
import time, json
import torch

results       = []
total_samples = 0

# JSON prefix forcing: every response must start with {"transcription":"
PREFIX   = '{"transcription":"'
prefix_ids = processor.tokenizer.encode(PREFIX, add_special_tokens=False)

# Suppress tool-call tokens that can appear in Qwen outputs
tool_call_ids = processor.tokenizer.encode('<tool_call>', add_special_tokens=False)
bad_words     = [tool_call_ids] if tool_call_ids else None

with open(EVAL_FILE) as f:
    eval_lines = [line.strip() for line in f if line.strip()]

print(f'Evaluating {len(eval_lines)} samples...')

for i, line in enumerate(eval_lines):
    sample   = json.loads(line)
    messages = sample['messages']

    # Reference from GPT assistant turn
    reference = get_reference(messages)
    if not reference:
        continue

    # Build prompt (user turn only)
    prompt_messages = [m for m in messages if m['role'] != 'assistant']
    images = extract_images(messages)

    prompt_text = processor.apply_chat_template(
        prompt_messages,
        tokenize=False,
        add_generation_prompt=True
    ) + PREFIX

    inputs = processor(
        text   = prompt_text,
        images = images if images else None,
        return_tensors = 'pt',
        padding        = True,
    ).to(model.device)

    t0 = time.time()
    with torch.inference_mode():
        output_ids = model.generate(
            **inputs,
            max_new_tokens        = 128,
            do_sample             = False,
            use_cache             = True,
            repetition_penalty    = 1.1,
            no_repeat_ngram_size  = 6,
            eos_token_id          = processor.tokenizer.eos_token_id,
            pad_token_id          = processor.tokenizer.pad_token_id,
            bad_words_ids         = bad_words,
        )
    elapsed = time.time() - t0

    # Decode only the newly generated tokens
    generated_ids = output_ids[0][inputs['input_ids'].shape[1]:]
    raw_output    = processor.tokenizer.decode(generated_ids, skip_special_tokens=True)

    # Prepend the forced prefix back before parsing
    full_output = PREFIX + raw_output
    hypothesis, is_valid_json = parse_output(full_output)

    results.append({
        'idx':            i,
        'reference':      reference,
        'hypothesis':     hypothesis,
        'is_valid_json':  is_valid_json,
        'inference_time': elapsed,
        'raw_output':     raw_output,
    })
    total_samples += 1

    if (i + 1) % 20 == 0:
        print(f'  [{i+1}/{len(eval_lines)}] elapsed so far: {sum(r["inference_time"] for r in results):.1f}s')

print(f'\nInference complete. {total_samples} samples processed.')

The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Evaluating 280 samples...
  [20/280] elapsed so far: 70.9s
  [40/280] elapsed so far: 155.3s
  [60/280] elapsed so far: 235.1s
  [80/280] elapsed so far: 311.6s
  [100/280] elapsed so far: 390.7s
  [120/280] elapsed so far: 464.5s
  [140/280] elapsed so far: 542.4s
  [160/280] elapsed so far: 622.0s
  [180/280] elapsed so far: 701.5s
  [200/280] elapsed so far: 772.4s
  [220/280] elapsed so far: 848.7s
  [240/280] elapsed so far: 925.7s
  [260/280] elapsed so far: 997.8s
  [280/280] elapsed so far: 1072.3s

Inference complete. 280 samples processed.


## Step 6.7 — Compute and report metrics

In [9]:
import json, os
from jiwer import cer, wer

references  = [r['reference']  for r in results if r['reference']]
hypotheses  = [r['hypothesis'] for r in results if r['reference']]
infer_times = [r['inference_time'] for r in results]

# CER / WER via jiwer (normalized to [0, 1])
avg_cer = round(cer(references, hypotheses), 4)
avg_wer = round(wer(references, hypotheses), 4)

# Exact match
exact_matches  = sum(1 for r, h in zip(references, hypotheses) if r == h)
exact_match_rt = round(exact_matches / len(references) * 100, 2)

# Valid JSON
valid_json     = sum(1 for r in results if r['is_valid_json'])
valid_json_rt  = round(valid_json / len(results) * 100, 2)

avg_inf_time   = round(sum(infer_times) / len(infer_times), 2)

print('=== Evaluation Results ===')
print(f'  Checkpoint       : {CHECKPOINT or "checkpoint-final"}')
print(f'  Num samples      : {total_samples}')
print(f'  CER              : {avg_cer}')
print(f'  WER              : {avg_wer}')
print(f'  Exact match rate : {exact_match_rt}%')
print(f'  Valid JSON rate  : {valid_json_rt}%')
print(f'  Avg infer time   : {avg_inf_time} s/sample')

# ── Save results ──────────────────────────────────────────────────────────
results_file_name = (
    f'eval_results_{CHECKPOINT}.json' if CHECKPOINT
    else 'eval_results_final.json'
)
results_path = f'{LOG_DIR}/{results_file_name}'

summary = {
    'checkpoint':         CHECKPOINT or 'checkpoint-final',
    'num_samples':        total_samples,
    'avg_cer':            avg_cer,
    'avg_wer':            avg_wer,
    'exact_match_rate':   exact_match_rt,
    'valid_json_rate':    valid_json_rt,
    'avg_inference_time': avg_inf_time,
    'per_sample':         results,
}

with open(results_path, 'w', encoding='utf-8') as f:
    json.dump(summary, f, indent=2, ensure_ascii=False)

print(f'\nResults saved to {results_path}')

=== Evaluation Results ===
  Checkpoint       : checkpoint-1120
  Num samples      : 280
  CER              : 0.3283
  WER              : 0.6554
  Exact match rate : 1.07%
  Valid JSON rate  : 92.14%
  Avg infer time   : 3.83 s/sample

Results saved to /content/drive/MyDrive/CUD files/NLP/NLP_Arabic_HTR_Project/logs/run-2/eval_results_checkpoint-1120.json


## Step 6.8 — Error analysis: worst predictions

Print the 5 samples with the highest per-sample CER to understand where the model fails.

In [11]:
from jiwer import cer as jiwer_cer

per_sample_cer = []
for r in results:
    ref = r['reference']
    hyp = r['hypothesis']
    if ref.strip():
        try:
            sample_cer = jiwer_cer([ref], [hyp])
        except Exception:
            sample_cer = 1.0
    else:
        sample_cer = 0.0
    per_sample_cer.append((sample_cer, r))

per_sample_cer.sort(key=lambda x: -x[0])

print('=== Top 5 Worst Predictions (by CER) ===')
for rank, (sample_cer, r) in enumerate(per_sample_cer[:5], 1):
    print(f'\n[{rank}] Sample {r["idx"]}  |  CER = {sample_cer:.4f}  |  Infer time = {r["inference_time"]:.2f}s')
    print(f'  REF : {r["reference"]}')
    print(f'  HYP : {r["hypothesis"]}')

print('\n=== Top 5 Best Predictions (by CER) ===')
for rank, (sample_cer, r) in enumerate(per_sample_cer[-5:][::-1], 1):
    print(f'\n[{rank}] Sample {r["idx"]}  |  CER = {sample_cer:.4f}')
    print(f'  REF : {r["reference"]}')
    print(f'  HYP : {r["hypothesis"]}')

=== Top 5 Worst Predictions (by CER) ===

[1] Sample 99  |  CER = 1.5000  |  Infer time = 2.61s
  REF : تأشيرة
  HYP : تُأَمِّنْهُ

[2] Sample 109  |  CER = 1.1250  |  Infer time = 1.19s
  REF : من يعترف
  HYP : من لكو لفته ،

[3] Sample 132  |  CER = 1.0370  |  Infer time = 5.50s
  REF : إنما اردنا البحث عن مورد يصور لنا احوال الحياة الحالية
  HYP : البحث عن مورد يصور لنا الحوادث الجيازية، إننا أردنا البحث عن موردي صورنا لحوادث الجيزة

[4] Sample 63  |  CER = 1.0000  |  Infer time = 1.24s
  REF : بودا فلوح
  HYP : لورد ا ميلافي،

[5] Sample 151  |  CER = 1.0000  |  Infer time = 5.91s
  REF : حابي يا عار، وخطها أكتبها ١٠/٢١؟
  HYP : حالي وكال ٢٠١٥ وتحفظها حتى سبتمبر ٢٠.٨/٢١٦ / عطلة

=== Top 5 Best Predictions (by CER) ===

[1] Sample 272  |  CER = 0.0000
  REF : أعتقد أن هذا غير صحيح في أماكن أخرى ولا يوجد مثله في كثير
  HYP : أعتقد أن هذا غير صحيح في أماكن أخرى ولا يوجد مثله في كثير

[2] Sample 255  |  CER = 0.0000
  REF : الرحان كانت تخرج في عهد الخليفة عثمان من بعض الجبال القريبة من

## Step 6.9 — Compare both checkpoints (re-run with CHECKPOINT = None for final)

Change `CHECKPOINT` in Step 6.1 to `None` and re-run Steps 6.4–6.7 to evaluate the
merged final adapter. The table below will let you compare them side by side once both
runs are complete.

Expected results from the AWS run:

| Checkpoint | CER | WER | Exact match | Valid JSON | Avg infer (s) |
|---|---|---|---|---|---|
| checkpoint-1120 | 1.2824 | 1.2767 | 0.0% | 100.0% | 4.52 |
| checkpoint-final (merged) | 1.4505 | 1.5765 | 0.0% | 100.0% | 3.52 |

> Note: CER/WER > 1.0 means the model is over-generating relative to the GPT reference.
> `jiwer` normalizes these correctly but values can still exceed 1.0 when the hypothesis
> is much longer than the reference. `checkpoint-1120` is the recommended pick.

based on error rates, Colab run is substantially better than the AWS run, and the numbers are not misleading.

The key difference is in how labels were constructed. AWS Run 2 masked nothing — `labels = input_ids.clone()` over the full 8192-token sequence, so during training the model was penalized for every token including image patches and the system prompt. This produced a model that learned the sequence structure but had weak transcription quality, reflected in CER/WER both above 1.0 (meaning the hypothesis was on average longer than the reference). Your Colab run inherited the same label approach, so the training loss numbers were higher in absolute terms — but the evaluation metric that matters is CER/WER on actual transcriptions, not cross-entropy during training.

The AWS CER of 1.28 means the model was generating roughly 28% more characters than the reference on average, often over-generating. Your CER of 0.33 means character-level errors are only 33% of the reference length — a genuinely better transcription. The best predictions (three exact matches at CER=0.0, including a long 15-word sentence) confirm the model is actually reading the images correctly on a meaningful fraction of samples.

The valid JSON rate dropping from 100% to 92% is worth noting — about 22 samples out of 280 produced malformed JSON. That is likely because the label masking difference affected how strongly the model learned the output format. It is not a serious problem for the paper but worth documenting in the Challenges section.

In short: CER 0.33 vs 1.28 — this model is better. The AWS numbers were inflated by the over-generation problem that was never fully resolved there.